# Évaluation — Sujet A : Frais médicaux d’un assureur santé
## Partie 1 : Régression linéaire multiple · Partie 2 : Random Forest

**Problématique :** expliquer et estimer les frais médicaux annuels (`charges`) à partir des caractéristiques des assurés. Le jeu demandé dans l'énoncé est *Medical Cost Personal Datasets* sur Kaggle.

**Barème :** exploration/corrélations (2), construction (3), interprétation des coefficients (3), évaluation/hypothèses (2), puis problème et choix du second modèle (2), explication mathématique (4), visualisation (2), mise en œuvre (2).

> Aucun résultat n'est inventé : les tableaux, métriques et graphiques sont calculés à l'exécution. Les associations ne prouvent pas une causalité. Ce notebook est pédagogique et ne doit pas servir seul à prendre des décisions réelles.

## 0. Préparer l'environnement
Exécutez les cellules dans l'ordre. Importez le CSV ou fichier Excel téléchargé depuis Kaggle lorsque demandé. Le prétraitement est ajusté sur l'entraînement uniquement afin d'éviter les fuites d'information.

In [ ]:
import sys, subprocess, importlib.util, os, warnings
warnings.filterwarnings("ignore")
if importlib.util.find_spec("statsmodels") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "statsmodels", "openpyxl", "xlrd"])
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns, statsmodels.api as sm
from google.colab import files
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, median_absolute_error
from sklearn.inspection import permutation_importance
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 100)
RANDOM_STATE = 42
print("Bibliothèques chargées.")

## 1. Charger les données
Le code accepte CSV, XLS ou XLSX. Vérifiez les noms de colonnes affichés et assurez-vous que la cible est bien `charges`.

In [ ]:
uploaded = files.upload()
if not uploaded:
    raise ValueError("Aucun fichier importé.")
filename = next(iter(uploaded))
ext = os.path.splitext(filename)[1].lower()
if ext == ".csv":
    try:
        df = pd.read_csv(filename)
        if df.shape[1] == 1: df = pd.read_csv(filename, sep=";")
    except UnicodeDecodeError:
        df = pd.read_csv(filename, encoding="latin-1")
elif ext in [".xls", ".xlsx"]:
    df = pd.read_excel(filename)
else:
    raise ValueError("Format non pris en charge : utilisez CSV, XLS ou XLSX.")
df.columns = [str(c).strip().lower().replace(" ", "_") for c in df.columns]
print("Fichier :", filename, "| Dimensions :", df.shape)
print("Colonnes :", df.columns.tolist())
display(df.head())

In [ ]:
TARGET = "charges"
if TARGET not in df.columns:
    raise ValueError(f"Colonne '{TARGET}' introuvable. Colonnes : {df.columns.tolist()}")
df[TARGET] = pd.to_numeric(df[TARGET], errors="coerce")
print("Types :"); display(df.dtypes.rename("type").to_frame())
print("Valeurs manquantes :"); display(df.isna().sum().sort_values(ascending=False).to_frame("manquantes"))
print("Lignes dupliquées :", int(df.duplicated().sum()))
display(df.describe(include="all").T)
df = df.dropna(subset=[TARGET]).copy()
for col in df.columns:
    if col != TARGET and df[col].dtype == "object":
        df[col] = df[col].astype("string").str.strip()
print("Dimensions après retrait des cibles manquantes :", df.shape)

## 2. Exploration des données et corrélations (2 points)

### 2.1 Distribution de `charges`
L'histogramme montre la forme de la distribution. Une asymétrie à droite ou des valeurs très élevées peuvent influencer l'ajustement linéaire.

In [ ]:
plt.figure(figsize=(8,5))
sns.histplot(df[TARGET], kde=True)
plt.title("Distribution des frais médicaux annuels (charges)")
plt.xlabel("Charges annuelles"); plt.ylabel("Nombre d'assurés")
plt.tight_layout(); plt.show()
print("Moyenne :", round(df[TARGET].mean(), 2))
print("Médiane :", round(df[TARGET].median(), 2))
print("Écart-type :", round(df[TARGET].std(), 2))

### 2.2 Comparer les charges selon les variables catégorielles
Les boîtes à moustaches comparent les distributions par groupe. Elles décrivent des associations, pas des effets causaux.

In [ ]:
X_explore = df.drop(columns=[TARGET])
cat_cols = X_explore.select_dtypes(include=["object", "string", "category", "bool"]).columns.tolist()
num_cols = X_explore.select_dtypes(include=np.number).columns.tolist()
print("Variables catégorielles :", cat_cols)
print("Variables numériques :", num_cols)
for col in cat_cols:
    if df[col].nunique(dropna=True) <= 20:
        plt.figure(figsize=(8,4))
        order = df.groupby(col, dropna=False)[TARGET].median().sort_values().index
        sns.boxplot(data=df, x=col, y=TARGET, order=order)
        plt.title(f"Charges selon {col}"); plt.xlabel(col); plt.ylabel("Charges annuelles")
        plt.xticks(rotation=20); plt.tight_layout(); plt.show()

### 2.3 Corrélations numériques
La corrélation de Pearson \(r\) mesure l'intensité et le sens d'une relation linéaire entre deux variables numériques, de -1 à +1. Elle ne détecte pas toutes les relations et ne prouve pas la causalité.

In [ ]:
num_df = df.select_dtypes(include=np.number)
if num_df.shape[1] >= 2:
    corr = num_df.corr()
    plt.figure(figsize=(8,6))
    sns.heatmap(corr, annot=True, cmap="coolwarm", center=0, fmt=".2f")
    plt.title("Matrice de corrélation des variables numériques")
    plt.tight_layout(); plt.show()
    if TARGET in corr:
        display(corr[TARGET].drop(TARGET).sort_values(key=lambda s: s.abs(), ascending=False)
                .to_frame("corrélation_avec_charges"))

**À compléter après exécution :**
- La distribution de `charges` est-elle symétrique ou asymétrique ?
- Quels groupes ont des médianes visiblement différentes ?
- Quelles variables numériques sont les plus corrélées à `charges` ?
- Voyez-vous des valeurs extrêmes ?
À l'oral, décrivez une observation réellement visible puis précisez la limite du graphique.

## 3. Préparer les données et construire la régression (3 points)

On sépare les observations en entraînement et test. Le modèle apprend sur l'entraînement ; le test mesure sa généralisation sur des observations non utilisées pour l'ajustement. Les variables catégorielles sont encodées par *one-hot encoding*. Les valeurs manquantes éventuelles sont imputées dans un pipeline.

In [ ]:
X = df.drop(columns=[TARGET]).copy()
y = df[TARGET].copy()
id_cols = [c for c in X.columns if c.lower() in {"id", "index", "customer_id", "client_id"}]
if id_cols:
    print("Identifiants exclus :", id_cols)
    X = X.drop(columns=id_cols)
numeric_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = [c for c in X.columns if c not in numeric_features]
print("Numériques :", numeric_features)
print("Catégorielles :", categorical_features)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)
try:
    encoder = OneHotEncoder(handle_unknown="ignore", drop="first", sparse_output=False)
except TypeError:
    encoder = OneHotEncoder(handle_unknown="ignore", drop="first", sparse=False)
numeric_pipe = Pipeline([("imputer", SimpleImputer(strategy="median"))])
categorical_pipe = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", encoder)])
preprocessor = ColumnTransformer([
    ("num", numeric_pipe, numeric_features),
    ("cat", categorical_pipe, categorical_features)
], remainder="drop", verbose_feature_names_out=False)
print("Entraînement :", X_train.shape, "| Test :", X_test.shape)

### 3.1 Formule de la régression linéaire multiple

\[
\widehat{charges}_i=\beta_0+\beta_1x_{i1}+\cdots+\beta_px_{ip}
\]

- \(β_0\) est la constante.
- \(β_j\) est la variation moyenne prédite des charges pour une unité supplémentaire de \(x_j\), à autres variables incluses constantes.
- Pour une variable catégorielle encodée, le coefficient est une différence par rapport à la catégorie de référence.

La méthode des moindres carrés choisit les coefficients qui minimisent :

\[
\sum_{i=1}^{n}(y_i-\widehat y_i)^2
\]

La relation moyenne est supposée linéaire dans les variables introduites. Cette hypothèse sera discutée à partir des résidus.

In [ ]:
linear_model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", LinearRegression())
])
linear_model.fit(X_train, y_train)
y_pred_linear = linear_model.predict(X_test)
print("Régression linéaire ajustée.")

## 4. Interpréter les coefficients (3 points)

Le signe du coefficient indique le sens de l'association prédite, conditionnellement aux autres variables du modèle. Le coefficient d'une catégorie se compare à la catégorie de référence. Les coefficients ne prouvent pas une causalité et peuvent être sensibles aux variables omises, aux valeurs extrêmes et à la multicolinéarité.

In [ ]:
fitted_prep = linear_model.named_steps["preprocessor"]
feature_names = fitted_prep.get_feature_names_out()
coef = linear_model.named_steps["regressor"].coef_
coef_table = pd.DataFrame({"variable": feature_names, "coefficient": coef})
display(coef_table.reindex(coef_table.coefficient.abs().sort_values(ascending=False).index))
print("Constante :", linear_model.named_steps["regressor"].intercept_)

### 4.1 Tableau OLS avec intervalles de confiance et p-values

`scikit-learn` fournit les coefficients prédictifs, mais pas directement les p-values. Le modèle OLS `statsmodels` ci-dessous fournit une table statistique complémentaire, calculée sur l'entraînement prétraité.

- `coef` : coefficient estimé.
- `P>|t|` : p-value du test \(H_0:β_j=0\), sous les hypothèses du modèle.
- `[0.025, 0.975]` : intervalle de confiance à 95 %.
- Une p-value faible ne mesure pas l'importance pratique et ne prouve pas une causalité. Les p-values usuelles peuvent être peu fiables en présence d'hétéroscédasticité.

In [ ]:
Xtr = fitted_prep.transform(X_train)
names = fitted_prep.get_feature_names_out()
Xtr_df = pd.DataFrame(Xtr, columns=names, index=X_train.index)
Xtr_df = sm.add_constant(Xtr_df, has_constant="add")
ols_result = sm.OLS(y_train, Xtr_df).fit()
display(ols_result.summary())

## 5. Évaluation et hypothèses (2 points)

Métriques sur le jeu de test :
- **MAE** \(=\frac{1}{n}\sum |y_i-\hat y_i|\) : erreur absolue moyenne, dans l'unité des charges.
- **RMSE** \(=\sqrt{\frac{1}{n}\sum(y_i-\hat y_i)^2}\) : pénalise davantage les grandes erreurs.
- **\(R^2\)** : comparaison à une prédiction constante par la moyenne du test ; il peut être négatif.
- **Erreur absolue médiane** : médiane des erreurs absolues, moins sensible à quelques grandes erreurs.

In [ ]:
def regression_metrics(y_true, y_pred, model_name):
    return {
        "Modèle": model_name,
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred),
        "Erreur_absolue_médiane": median_absolute_error(y_true, y_pred)
    }
linear_metrics = regression_metrics(y_test, y_pred_linear, "Régression linéaire")
display(pd.DataFrame([linear_metrics]).set_index("Modèle").round(3))
plt.figure(figsize=(7,6))
sns.scatterplot(x=y_test, y=y_pred_linear, alpha=.65)
lo=min(float(y_test.min()), float(np.min(y_pred_linear)))
hi=max(float(y_test.max()), float(np.max(y_pred_linear)))
plt.plot([lo,hi],[lo,hi],"--")
plt.xlabel("Charges observées"); plt.ylabel("Charges prédites")
plt.title("Régression linéaire : observé vs prédit")
plt.tight_layout(); plt.show()

### 5.1 Résidus et hypothèses

Un résidu est \(e_i=y_i-\hat y_i\). Les diagnostics à discuter :
1. **Linéarité** : la forme linéaire représente-t-elle raisonnablement la relation moyenne ?
2. **Indépendance** : les erreurs sont-elles indépendantes ? Cela dépend notamment du plan d'échantillonnage.
3. **Homoscedasticité** : la variance des erreurs est-elle à peu près constante ?
4. **Normalité des résidus** : surtout utile pour certains tests et intervalles, notamment en petit échantillon.
5. **Multicolinéarité** : des prédicteurs corrélés peuvent rendre les coefficients instables.
6. **Valeurs influentes** : certaines observations extrêmes peuvent peser fortement sur l'ajustement.

Les graphiques donnent des indices, mais ne prouvent pas à eux seuls toutes les hypothèses.

In [ ]:
resid = y_test.to_numpy() - np.asarray(y_pred_linear)
fig, axes = plt.subplots(1,2,figsize=(13,5))
sns.scatterplot(x=y_pred_linear, y=resid, alpha=.65, ax=axes[0])
axes[0].axhline(0, linestyle="--")
axes[0].set_title("Résidus vs valeurs prédites")
axes[0].set_xlabel("Charges prédites"); axes[0].set_ylabel("Résidu")
sns.histplot(resid, kde=True, ax=axes[1])
axes[1].set_title("Distribution des résidus"); axes[1].set_xlabel("Résidu")
plt.tight_layout(); plt.show()
sm.qqplot(resid, line="45", fit=True)
plt.title("QQ-plot des résidus du test")
plt.tight_layout(); plt.show()

**À compléter après exécution :** notez les valeurs de MAE, RMSE et R² ; expliquez ce que signifie la MAE dans l'unité monétaire ; décrivez si les résidus sont centrés autour de zéro, si leur dispersion semble changer avec les prédictions et si le QQ-plot montre des écarts marqués. Évitez d'affirmer que les hypothèses sont toutes vérifiées sur la seule base des graphiques.

# Partie 2 — Random Forest Regressor

## 6. Problème métier et choix du modèle (2 points)

Le problème reste l'estimation des frais annuels. Une régression linéaire impose une forme linéaire dans les variables introduites. Les coûts peuvent cependant présenter des seuils et interactions. Une **Random Forest Regressor** combine plusieurs arbres de décision et peut représenter des relations non linéaires sans imposer une équation linéaire globale.

Limites : le modèle est moins directement interprétable et extrapole difficilement au-delà des valeurs de cible observées dans l'entraînement. Il faut le comparer à la régression sur le même jeu de test.

## 7. Fonctionnement mathématique de la Random Forest (4 points)

### Arbre de régression
Un arbre partitionne les données en régions. Dans une feuille \(R_m\), la prédiction est généralement la moyenne des valeurs d'entraînement de cette région :

\[
\hat c_m=\frac{1}{N_m}\sum_{i:x_i\in R_m} y_i
\]

Les coupures cherchent à réduire l'erreur quadratique au sein des régions.

### Forêt aléatoire
Pour chacun des \(B\) arbres :
1. tirer un échantillon bootstrap des lignes (tirage avec remise) ;
2. construire un arbre en considérant un sous-ensemble aléatoire de variables à chaque séparation ;
3. obtenir une prédiction \(T_b(x)\).

La prédiction finale est la moyenne :

\[
\hat f(x)=\frac{1}{B}\sum_{b=1}^{B}T_b(x)
\]

La diversité entre les arbres peut réduire la variance de l'ensemble. Le gain est plus faible si les arbres commettent des erreurs très similaires.

**Paramètres importants :** `n_estimators` (nombre d'arbres), `max_depth` (profondeur), `min_samples_leaf` (taille minimale des feuilles), `max_features` (variables candidates par coupure), `random_state` (reproductibilité). La forêt n'exige pas de normalisation des variables numériques.

### 7.1 Schéma conceptuel (2 points)
Ce schéma illustre le principe général, pas un arbre réellement entraîné sur les données.

In [ ]:
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
fig, ax = plt.subplots(figsize=(12,5))
ax.set_xlim(0,12); ax.set_ylim(0,6); ax.axis("off")
def box(x,y,w,h,label):
    ax.add_patch(FancyBboxPatch((x,y),w,h,boxstyle="round,pad=0.12",facecolor="white",linewidth=1.2))
    ax.text(x+w/2,y+h/2,label,ha="center",va="center",fontsize=10)
def arrow(a,b,c,d):
    ax.add_patch(FancyArrowPatch((a,b),(c,d),arrowstyle="->",mutation_scale=14))
box(.3,2.3,2,1.1,"Données\nd'entraînement")
box(3,4.2,2,1,"Bootstrap 1\n+ arbre 1")
box(3,2.3,2,1,"Bootstrap 2\n+ arbre 2")
box(3,.4,2,1,"Bootstrap B\n+ arbre B")
box(6,4.2,1.8,1,"Prédiction T₁(x)")
box(6,2.3,1.8,1,"Prédiction T₂(x)")
box(6,.4,1.8,1,"Prédiction Tᴮ(x)")
box(9,2.3,2.5,1.1,"Moyenne des arbres\nŷ(x) = Σ Tᵦ(x) / B")
for yy in [4.7,2.8,.9]: arrow(2.3,2.85,3,yy)
for yy in [4.7,2.8,.9]: arrow(5,yy,6,yy)
arrow(7.8,4.7,9,3.1); arrow(7.8,2.8,9,2.85); arrow(7.8,.9,9,2.55)
ax.set_title("Principe d'une Random Forest en régression", fontsize=14)
plt.tight_layout(); plt.show()

## 8. Mise en œuvre de la Random Forest (2 points)

Les paramètres ci-dessous sont un point de départ pédagogique, pas une garantie d'optimalité. Le prétraitement est appris sur l'entraînement uniquement. Le test reste réservé à l'évaluation finale.

In [ ]:
rf_model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", RandomForestRegressor(
        n_estimators=300, max_depth=None, min_samples_leaf=2,
        max_features=1.0, random_state=RANDOM_STATE, n_jobs=-1
    ))
])
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)
print("Random Forest ajustée.")

## 9. Comparer les modèles

MAE et RMSE plus faibles indiquent des erreurs plus faibles. Un R² plus élevé est préférable selon cette métrique. Examinez plusieurs indicateurs : un modèle peut réduire certaines grandes erreurs sans beaucoup changer l'erreur absolue moyenne.

In [ ]:
rf_metrics = regression_metrics(y_test, y_pred_rf, "Random Forest")
comparison = pd.DataFrame([linear_metrics, rf_metrics]).set_index("Modèle")
display(comparison.round(3))
comparison[["MAE","RMSE"]].plot(kind="bar", figsize=(8,5))
plt.title("Comparaison des erreurs sur le jeu de test")
plt.ylabel("Erreur (unité de charges)"); plt.xticks(rotation=0)
plt.tight_layout(); plt.show()
comparison[["R2"]].plot(kind="bar", legend=False, figsize=(6,4))
plt.title("Comparaison du R² sur le jeu de test"); plt.ylabel("R²")
plt.xticks(rotation=0); plt.tight_layout(); plt.show()
plt.figure(figsize=(7,6))
sns.scatterplot(x=y_test, y=y_pred_rf, alpha=.65)
lo=min(float(y_test.min()), float(np.min(y_pred_rf)))
hi=max(float(y_test.max()), float(np.max(y_pred_rf)))
plt.plot([lo,hi],[lo,hi],"--")
plt.xlabel("Charges observées"); plt.ylabel("Charges prédites")
plt.title("Random Forest : observé vs prédit")
plt.tight_layout(); plt.show()

## 10. Importance des variables (complément d'interprétation)

L'importance par permutation mesure la dégradation de la performance lorsque les valeurs d'une variable sont mélangées. Une importance élevée signifie que le modèle s'appuie sur cette variable pour prédire ; elle ne prouve ni causalité ni direction d'effet. Des variables corrélées peuvent se partager l'importance.

In [ ]:
perm = permutation_importance(
    rf_model, X_test, y_test, n_repeats=10, random_state=RANDOM_STATE,
    scoring="neg_mean_absolute_error", n_jobs=-1
)
importance = pd.DataFrame({
    "variable": X_test.columns,
    "importance_moyenne": perm.importances_mean,
    "ecart_type": perm.importances_std
}).sort_values("importance_moyenne", ascending=False)
display(importance)
plt.figure(figsize=(8,max(4,.4*len(importance))))
sns.barplot(data=importance, x="importance_moyenne", y="variable")
plt.title("Importance par permutation (MAE)")
plt.xlabel("Dégradation moyenne du score après mélange"); plt.ylabel("Variable")
plt.tight_layout(); plt.show()

## 11. Conclusion à rédiger après exécution

Remplacez les éléments entre crochets par les valeurs réellement obtenues.

- **Exploration :** la distribution de `charges` [décrire le graphique] ; les variables/groupes qui présentent les associations descriptives les plus visibles sont [résultats observés].
- **Régression linéaire :** MAE = [valeur], RMSE = [valeur], R² = [valeur]. La MAE représente une erreur absolue moyenne d'environ [valeur] dans l'unité de `charges`.
- **Hypothèses :** les graphiques de résidus suggèrent [observation], avec les limites suivantes : [limites].
- **Random Forest :** MAE = [valeur], RMSE = [valeur], R² = [valeur]. Par rapport à la régression linéaire, elle [améliore / détériore / donne des résultats comparables] selon les métriques.
- **Conclusion métier :** le modèle retenu est [modèle], car [métriques et objectif]. Avant usage réel, une validation externe, une analyse des biais, une gouvernance des données et une surveillance dans le temps seraient nécessaires.

## 12. Questions possibles à l'oral

1. **Pourquoi explorer les données ?** Pour comprendre les types, distributions, valeurs manquantes, extrêmes et associations potentielles.
2. **Pourquoi séparer entraînement et test ?** Pour estimer la généralisation sur des observations non utilisées pendant l'ajustement.
3. **Pourquoi encoder les variables catégorielles ?** Les modèles attendent des entrées numériques ; le one-hot encoding crée des indicatrices sans imposer un ordre arbitraire.
4. **Que signifie un coefficient positif ?** Une hausse de la prédiction moyenne conditionnellement aux autres variables, ou une différence par rapport à la catégorie de référence.
5. **Que minimise la méthode des moindres carrés ?** La somme des carrés des résidus sur l'entraînement.
6. **Différence entre MAE et RMSE ?** La RMSE pénalise davantage les erreurs élevées car les erreurs sont mises au carré.
7. **Que signifie R² ?** La performance relative à une prédiction constante par la moyenne ; il peut être négatif sur le test.
8. **Pourquoi examiner les résidus ?** Pour repérer des motifs suggérant une forme mal spécifiée, une variance non constante ou des valeurs influentes.
9. **Comment fonctionne un arbre de régression ?** Il partitionne l'espace des variables et prédit généralement la moyenne des cibles dans chaque feuille.
10. **Pourquoi une forêt contient plusieurs arbres ?** Pour combiner des arbres diversifiés et réduire la variance.
11. **Pourquoi le bootstrap ?** Chaque arbre apprend sur un échantillon tiré avec remise, ce qui diversifie les arbres.
12. **Pourquoi choisir des variables aléatoires à chaque coupure ?** Pour réduire la similarité entre les arbres.
13. **La Random Forest est-elle toujours meilleure ?** Non ; la performance dépend des données, du bruit, des paramètres et de la métrique.
14. **Que mesure l'importance par permutation ?** La dégradation du score quand une variable est mélangée ; ce n'est pas un effet causal.
15. **Quelles sont les limites ?** Variables omises, jeu de données limité, un seul découpage, valeurs extrêmes, hypothèses imparfaites et absence de validation externe.

## 13. Checklist avant partage avec l'enseignant

- [ ] Toutes les cellules s'exécutent dans l'ordre.
- [ ] Le fichier chargé est le bon et la cible est `charges`.
- [ ] Les résultats affichés proviennent de l'exécution réelle.
- [ ] Les métriques sont calculées sur le même jeu de test pour les deux modèles.
- [ ] Les coefficients et graphiques sont interprétés avec prudence.
- [ ] Les hypothèses et limites sont discutées sans prétendre qu'un graphique prouve tout.
- [ ] Les deux membres du groupe savent expliquer les formules et les paramètres.
- [ ] Le notebook est partagé avant l'heure limite.